# IFEval on ScreamingFace: one family, four experiments

IFEval (arXiv:2311.07911) is 541 prompts with machine-checkable constraints — word
counts, forbidden punctuation, required sections. The Engine grades every response with
a deterministic verifier: **no judge model in the grading path, zero grading cost**.

There is one IFEval family with three explicit Variants:

- `ifeval` — one shot. A solo Model answers once; a Fusion's members answer and its
  synthesizer **blends** them into one new answer. The blend is checked.
- `ifeval/self-corrective` — up to three attempts. The whole Candidate reads the
  checker's violations, **writes its own feedback, and retries**.
- `ifeval/verifying-ensemble` — the current verifying ensemble implementation based on
  Skurikhin et al. (https://openreview.net/forum?id=XSIYfTm2h7): every direct Fusion
  member is checked individually, and the **synthesizer acts as JUDGE** — it picks a
  passing answer word-for-word, or turns the violations into coaching when nobody
  passed. It never writes the answer on this exam.

One rule to remember: **the synthesizer plays two roles.** Blender on `ifeval`,
judge on `ifeval/verifying-ensemble`.

## Before running

AI Gateway on `127.0.0.1:9105`, Engine on `127.0.0.1:9108`. From `packages/screamingface/`:

```bash
just stack-prepare   # once — downloads the pinned benchmark cases
just stack-up        # gateway :9105 + engine :9108 (logs: just stack-logs)
```

In [ ]:
import screamingface as sf

In [ ]:
sf.connect()

## The Candidates

Two models and one Fusion. The Fusion's synthesizer is also a member — the
winning ensemble of Skurikhin et al. ([Ens-1]) is shaped exactly like this: two
members, with the judge doubling as one of them.

In [ ]:
kimi = sf.Model("openrouter/moonshotai/kimi-k3", params={"max_tokens": 4096})
haiku = sf.Model("openrouter/anthropic/claude-haiku-4.5")

fusion = sf.Fusion(
    [kimi, haiku],
    name="kimi-haiku",
    synthesizer="openrouter/moonshotai/kimi-k3",
)
fusion

## ① Baseline — one model, one shot

Comparable to published IFEval numbers.

In [ ]:
canonical_1_model = sf.evaluate(
    kimi,
    benchmark="ifeval",
    limit=3,
    progress=False,
)
canonical_1_model

## ② Does blending preserve instructions?

The synthesizer writes one NEW answer from the members' answers — new text the checker
never saw. A blend can break a constraint every member satisfied (add a comma, drop a
section). This cell measures that risk.

In [ ]:
canonical_fusion = sf.evaluate(
    fusion,
    benchmark="ifeval",
    limit=3,
    progress=False,
)
canonical_fusion

## ③ Can a model correct itself?

The ablation the paper never ran: {solo + feedback loop}. The model answers, the
checker reports violations, the model writes its own feedback and retries — up to
three attempts, earliest pass wins.

Cost: five model calls per case (three answers + two self-feedback authorings), all
unrolled.

In [ ]:
iterative_1_model = sf.evaluate(
    kimi,
    benchmark="ifeval/self-corrective",
    limit=3,
    progress=False,
)
iterative_1_model

## ④ The verifying ensemble (the paper's protocol)

Members answer, the checker checks **each draft individually**, and the synthesizer —
acting as judge here — picks a passing answer verbatim, or coaches everyone and retries
when nobody passed. A judge cannot break a constraint a member satisfied, because it
never rewrites the winning text.

Choose a synthesizer that reliably answers tersely: a judge reply that is not a bare
letter gets no vote (the deterministic passers-first rule decides instead), and the
synthesizer inherits provider-default params on this exam.

In [ ]:
iterative_fusion = sf.evaluate(
    fusion,
    benchmark="ifeval/verifying-ensemble",
    limit=3,
    progress=False,
)
iterative_fusion

## Reading the four scores

- ① vs ② — did blending help or hurt instruction-following?
- ① vs ③ — how much does a feedback loop help one model?
- ③ vs ④ — self-correction vs ensemble correction, same loop, same exam.
- ② vs ④ — blend-then-check vs check-then-select.

Cost note: the iterative-correction exam has no early stop yet — all three attempts
always run (and the solo shape adds two self-feedback calls), so its token totals
overstate a stop-on-success system. Compare scores freely within a column; never
compare our costs to the paper's.

In [ ]:
{
    name: {
        "score": report.candidates[0].score,
        "output_tokens": report.usage.output_tokens,
    }
    for name, report in {
        "① ifeval · kimi": canonical_1_model,
        "② ifeval · fusion": canonical_fusion,
        "③ iterative-correction · kimi": iterative_1_model,
        "④ iterative-correction · fusion": iterative_fusion,
    }.items()
}